# 藥物原料、feeder、contextual data對應
1. API：原料為Acetaminophen powder(APAP_P)、feed by PD3
2. MGST：原料為Magnesium stearate(Mag St)、feed by PD7
3. Loctose：原料為Spray dried lactose (L316FF)、feed by PD2
4. Ac-di-sol：Sodium croscarmellose (Ac-di-sol)、feed by PD5
5. Avicel PH102：原料為Microcrystalline cellulose, Avicel PH102，會有兩個批號同時供應、feed by PD1 and PD4

In [2]:
import pandas as pd

In [3]:
# machine data
liw_feeders_1 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 1.csv")
liw_feeders_2 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 2.csv")
blenders = pd.read_csv("MSOM data external sharing\Machine data\Blenders.csv")
pressor = pd.read_csv("MSOM data external sharing\Machine data\Tablet Press.csv")
temperature = pd.read_csv("MSOM data external sharing\Machine data\Temperature.csv")
humidity = pd.read_csv("MSOM data external sharing\Machine data\Humidity.csv")

C:\Users\User\AppData\Local\Temp\ipykernel_39900\4085920741.py:2: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_1 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 1.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_39900\4085920741.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  liw_feeders_2 = pd.read_csv("MSOM data external sharing\Machine data\LiW Feeders 2.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_39900\4085920741.py:4: DtypeWarning: Columns (1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  blenders = pd.read_csv("MSOM data external sharing\Machine data\Blenders.csv")
C:\Users\User\AppData\Local\Temp\ipykernel_39900\4085920741.py:5: DtypeWarning: Columns (1

In [4]:
# concat liw_feeder1&2
liw_feeder = pd.concat([liw_feeders_1, liw_feeders_2], axis=1)
liw_feeder = liw_feeder.loc[:, ~liw_feeder.columns.str.contains("Unnamed:")]

In [5]:
#日期處理
def parse_mixed_date(date_str):
    try:
        # 嘗試美式日期
        return pd.to_datetime(date_str, format="%m/%d/%Y %H:%M")
    except ValueError:
        try:
            # 如果失敗，改用歐洲格式
            return pd.to_datetime(date_str, format="%d/%m/%Y %H:%M")
        except:
            return pd.NaT
        

liw_feeder["TimeStamp"] = liw_feeder["TimeStamp"].apply(parse_mixed_date)
blenders["TimeStamp"] = blenders["TimeStamp"].apply(parse_mixed_date)
pressor["TimeStamp"] = pressor["TimeStamp"].apply(parse_mixed_date)


liw_feeder = liw_feeder.dropna(subset=["TimeStamp"], how="any")
blenders = blenders.dropna(subset=['TimeStamp'], how="any")
pressor = pressor.dropna(subset=['TimeStamp'], how="any")

In [70]:
# contextual quality
logbook = pd.read_excel("MSOM data external sharing\Contextual quality data\Logbook Long Run Days.xlsx")
content = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Content Uniformity.xlsx", header=1)
material_property = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Material Properties.xlsx")
tablet_property = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Tablet Properties and Drum Change.xlsx", sheet_name="Tablet properties", header=1)
drum_change = pd.read_excel("MSOM data external sharing\Contextual quality data\RM Tablet Properties and Drum Change.xlsx", sheet_name="Raw Material Drum change", header=1)

In [7]:
# tablet_property 處理
tablet_property = tablet_property.dropna(axis=0, how="all")

In [140]:
# drum_change 處理
api = drum_change.iloc[1:, 0:4].dropna(axis=0, how="all")
mgst = drum_change.iloc[1:, 4:8].dropna(axis=0, how="all")
lactose = drum_change.iloc[1:, 8:12].dropna(axis=0, how="all")
Ac_Di_Sol = drum_change.iloc[1:, 12:16].dropna(axis=0, how="all")
Avicel_102_PD1 = drum_change.iloc[1:, 16:20].dropna(axis=0, how="all")
Avicel_102_PD4 = drum_change.iloc[1:, 20:24].dropna(axis=0, how="all")

api = api[api["Est Refill time"] != "missing"]
mgst = mgst.rename(columns={"Date/Time.1": "Date/Time", "Lot.1":"Lot"})
lactose = lactose.rename(columns={"Date/Time.2": "Date/Time"})
Ac_Di_Sol = Ac_Di_Sol.rename(columns={"Date/Time.3": "Date/Time", "Drum #.1": "Drum #"})
Avicel_102_PD1 = Avicel_102_PD1.rename(columns={"Date/Time.4": "Date/Time", "Drum #.2": "Drum #"})
Avicel_102_PD4 = Avicel_102_PD4.rename(columns={"Date/Time.5": "Date/Time", "Drum #.3": "Drum #"})

In [141]:
# first day
# machine data
time_till_first_day = 60 * 60 * 15 + 60 * 17 - 5
liw_feeder_first_day = liw_feeder[:time_till_first_day]
blenders_first_day = blenders[:time_till_first_day]
pressor_first_day = pressor[:time_till_first_day]

# contextual data
api_first_day = api[:7]
mgst_first_day = mgst[:6]
lactose_first_day = lactose[:3]
Ac_Di_Sol_first_day = Ac_Di_Sol[:2]
Avicel_102_PD1_first_day = Avicel_102_PD1[:4]
Avicel_102_PD4_first_day = Avicel_102_PD4[:4]

In [142]:
data_set = {
    "API": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "MGST": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Lactose": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Ac_Di_Sol": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Avicel_102_PD1": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    },
    "Avicel_102_PD4": {
        "liw_feeder": {},
        "blenders": {},
        "pressor": {}
    }
}

In [143]:
# Based on the sampling interval of output type, find the data points of liw_feeder、blenders、pressor in each time interval.
# 設定時間區間
def time(i: int, input: pd.DataFrame):
    if i == -1: 
        return f'Before {input.iloc[i+1]["Date/Time"]}'
    
    if i == len(input) - 1: 
        return f'After {input.iloc[i-1]["Date/Time"]}'
    
    return f'{input.iloc[i]["Date/Time"]} to {input.iloc[i+1]["Date/Time"]}'

# 根據時間區間確定那些process data應該要放入
def data_within_time(i: int, input: pd.DataFrame, process: pd.DataFrame):
    # 在input的第一個時間之前的process data
    if i == -1:
        return process[process["TimeStamp"] <= input.iloc[i+1]["Date/Time"]]
    # 在input的最後一個時間之後的process data
    if i == len(input) - 1:
        return process[process["TimeStamp"] >= input.iloc[i-1]["Date/Time"]]

    return process[(process["TimeStamp"] > input.iloc[i]["Date/Time"]) & (process["TimeStamp"] < input.iloc[i+1]["Date/Time"])]

# 標記record的batch
def set_process_data_batch(df: pd.DataFrame, input_key:str, process_key:str, input: pd.DataFrame, process: pd.DataFrame):
    batch_name = "Drum #"
    

    # 如果是API或MGST的話就將batch_name換為Lot
    if input_key == "API" or input_key == "MGST":
        batch_name = "Lot"

    print(input_key, batch_name)
    batchs = input[batch_name].unique()
    
    keys = list(df[input_key][process_key].keys())

    # init batch column
    for key in keys:
        for batch in batchs:
            df[input_key][process_key][key][batch] = 0

        # 將batch分配到process表格中
        for i, row in input.iterrows():
            start_time = row["Date/Time"]
            next_time = input.iloc[i+1]["Date/Time"] if i + 1 < len(input) else pd.Timestamp.max
            
            batch = row[batch_name]

            #如果時間在範圍內，就將batch標記為1
            df[input_key][process_key][key].loc[(df[input_key][process_key][key]["TimeStamp"] >= start_time) & (df[input_key][process_key][key]["TimeStamp"] < next_time), batch] = 1

In [144]:
# 依據input與process將data區分
input_keys = list(data_set.keys())
process_keys = list(data_set['API'].keys())

inputs = [api_first_day, mgst_first_day, lactose_first_day, Ac_Di_Sol_first_day, Avicel_102_PD1_first_day, Avicel_102_PD4_first_day]
process = [liw_feeder_first_day, blenders_first_day, pressor_first_day]

for i, input_key in enumerate(input_keys):
    for j, process_key in enumerate(process_keys):
        k = -1
        while k < len(inputs[i]) - 1:
            data_set[input_key][process_key][time(k, inputs[i])] = data_within_time(k, inputs[i], process[j])
            k += 1

for i, input_key in enumerate(input_keys):
    for j, process_key in enumerate(process_keys):
        set_process_data_batch(data_set, input_key, process_key, inputs[i], process[j])

API Lot
API Lot
API Lot
MGST Lot
MGST Lot
MGST Lot
Lactose Drum #
Lactose Drum #
Lactose Drum #
Ac_Di_Sol Drum #
Ac_Di_Sol Drum #
Ac_Di_Sol Drum #
Avicel_102_PD1 Drum #
Avicel_102_PD1 Drum #
Avicel_102_PD1 Drum #
Avicel_102_PD4 Drum #
Avicel_102_PD4 Drum #
Avicel_102_PD4 Drum #


C:\Users\User\AppData\Local\Temp\ipykernel_39900\2830093123.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[input_key][process_key][key][batch] = 0
C:\Users\User\AppData\Local\Temp\ipykernel_39900\2830093123.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[input_key][process_key][key][batch] = 0
C:\Users\User\AppData\Local\Temp\ipykernel_39900\2830093123.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

In [145]:
# 因為最內層的key是由inputs決定，所以process用甚麼都沒關係
api_keys = list(data_set["API"]["liw_feeder"].keys())
mgst_keys = list(data_set["MGST"]["liw_feeder"].keys())
lactose_keys = list(data_set['Lactose']["liw_feeder"].keys())
Ac_Di_Sol_keys = list(data_set['Ac_Di_Sol']["liw_feeder"].keys())
Avicel_102_PD1_keys = list(data_set['Avicel_102_PD1']["liw_feeder"].keys())
Avicel_102_PD4_keys = list(data_set['Avicel_102_PD4']["liw_feeder"].keys())

In [146]:
# 標記contextual record的batch
def set_process_data_batch(df: pd.DataFrame, input_key:str, process_key:str, input: pd.DataFrame, process: pd.DataFrame):
    batch_name = "Drum #"
    
    # 如果是API或MGST的話就將batch_name換為Lot
    if input_key == "API" or input_key == "MGST":
        batch_name = "Lot"

    batchs = input[batch_name].unique()
    
    # init batch column
    for batch in batchs:
        process[batch] = 0

    # 將batch分配到process表格中
    for i, row in input.iterrows():
        start_time = row["Date/Time"]
        next_time = input.iloc[i+1]["Date/Time"] if i + 1 < len(input) else pd.Timestamp.max
        batch = row[batch_name]

        #如果時間在範圍內，就將batch標記為1
        process.loc[(process["TimeStamp"] >= start_time) & (process["TimeStamp"] < next_time), batch] = 1

    print(process)

In [147]:
content = content.iloc[:, :-1]

content_first_day = content[:26]
content_first_day.tail()

,Time,Sample,Tablet 1,Tablet 2,Tablet 3,Tablet 4,Tablet 5,Tablet 6,Tablet 7,Tablet 8
21,2018-01-12 21:46:00,22,10.18999,10.291230,10.38701,10.25019,10.44999,10.249130,10.34573,10.38087
22,2018-01-12 22:16:00,23,10.26376,9.993997,10.04832,10.01869,10.13186,10.141120,10.07814,10.06360
23,2018-01-12 22:46:00,24,10.23955,10.177670,10.06189,10.22093,10.27618,9.968208,10.14547,10.17240
24,2018-01-12 23:16:00,25,10.03864,10.163840,9.955503,10.1653,10.26333,10.165240,10.23519,10.05947
25,2018-01-12 23:46:00,26,10.2953,9.881690,10.21047,10.25368,9.955813,10.182410,10.22987,10.16079
